In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.14 STEP 1 — PRICING & DISCOUNT BASELINE ANALYSIS
# ============================================================

BASE_DIR = Path.cwd().parent

SALES_DIR = BASE_DIR / "data" / "synthetic"
PRODUCTS_FILE = SALES_DIR / "products_master.csv"

# ------------------------------------------------------------
# Load product master
# ------------------------------------------------------------

products = pd.read_csv(PRODUCTS_FILE)

print("02.14 STEP 1 — PRICING & DISCOUNT BASELINE ANALYSIS")
print("=" * 70)

print("\nProduct master rows:", len(products))

# ------------------------------------------------------------
# Load all sales files
# ------------------------------------------------------------

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

print("Sales files loaded:", len(sales_files))
print("Sales rows:", len(sales))

# ------------------------------------------------------------
# Basic pricing fields
# ------------------------------------------------------------

sales["unit_selling_price"] = pd.to_numeric(
    sales["unit_selling_price"],
    errors="coerce"
)

sales["discount_pct"] = pd.to_numeric(
    sales["discount_pct"],
    errors="coerce"
)

# ------------------------------------------------------------
# Merge product pricing information
# ------------------------------------------------------------

product_pricing = products[
    [
        "product_id",
        "category",
        "unit_price",
        "cost_price",
        "profit_margin_pct",
        "status"
    ]
].copy()

sales_pricing = sales.merge(
    product_pricing,
    on="product_id",
    how="left"
)

# ------------------------------------------------------------
# Calculate pricing metrics
# ------------------------------------------------------------

sales_pricing["price_vs_list_pct"] = (
    sales_pricing["unit_selling_price"]
    / sales_pricing["unit_price"]
    * 100
)

sales_pricing["gross_sales_value"] = (
    sales_pricing["quantity"]
    * sales_pricing["unit_selling_price"]
)

sales_pricing["discount_value"] = (
    sales_pricing["gross_sales_value"]
    * sales_pricing["discount_pct"]
    / 100
)

sales_pricing["net_sales_value"] = (
    sales_pricing["gross_sales_value"]
    - sales_pricing["discount_value"]
)

# ============================================================
# PRICING VALIDATION
# ============================================================

print("\nPricing Validation")
print("=" * 70)

print(
    "Missing unit selling prices:",
    sales_pricing["unit_selling_price"].isna().sum()
)

print(
    "Non-positive unit selling prices:",
    (sales_pricing["unit_selling_price"] <= 0).sum()
)

print(
    "Missing product list prices:",
    sales_pricing["unit_price"].isna().sum()
)

print(
    "Non-positive product list prices:",
    (sales_pricing["unit_price"] <= 0).sum()
)

print(
    "Missing product costs:",
    sales_pricing["cost_price"].isna().sum()
)

print(
    "Non-positive product costs:",
    (sales_pricing["cost_price"] <= 0).sum()
)

print(
    "Missing discounts:",
    sales_pricing["discount_pct"].isna().sum()
)

print(
    "Invalid discounts:",
    (
        (sales_pricing["discount_pct"] < 0)
        |
        (sales_pricing["discount_pct"] > 100)
    ).sum()
)

# ============================================================
# DISCOUNT BAND ANALYSIS
# ============================================================

def discount_band(value):

    if value == 0:
        return "0%"

    elif value <= 5:
        return "1%-5%"

    elif value <= 10:
        return ">5%-10%"

    elif value <= 15:
        return ">10%-15%"

    elif value <= 20:
        return ">15%-20%"

    elif value <= 30:
        return ">20%-30%"

    else:
        return ">30%-60%"

sales_pricing["discount_band"] = (
    sales_pricing["discount_pct"]
    .apply(discount_band)
)

discount_summary = (
    sales_pricing
    .groupby(
        "discount_band",
        as_index=False
    )
    .agg(
        sales_lines=("sales_line_id", "count"),
        average_discount=("discount_pct", "mean")
    )
)

discount_summary["share_pct"] = (
    discount_summary["sales_lines"]
    / discount_summary["sales_lines"].sum()
    * 100
)

band_order = [
    "0%",
    "1%-5%",
    ">5%-10%",
    ">10%-15%",
    ">15%-20%",
    ">20%-30%",
    ">30%-60%"
]

discount_summary["band_order"] = (
    discount_summary["discount_band"]
    .map(
        {
            band: i
            for i, band in enumerate(band_order)
        }
    )
)

discount_summary = (
    discount_summary
    .sort_values("band_order")
    .drop(columns="band_order")
)

print("\nDiscount Band Distribution")
print("=" * 70)

print(
    discount_summary.to_string(index=False)
)

# ============================================================
# PROMOTION VS NON-PROMOTION
# ============================================================

sales_pricing["promotion_flag"] = (
    sales_pricing["promotion_id"]
    .notna()
    &
    (
        sales_pricing["promotion_id"]
        .astype(str)
        .str.strip()
        != ""
    )
    &
    (
        sales_pricing["promotion_id"]
        .astype(str)
        .str.lower()
        != "nan"
    )
)

promotion_summary = (
    sales_pricing
    .groupby(
        "promotion_flag",
        as_index=False
    )
    .agg(
        sales_lines=("sales_line_id", "count"),
        average_discount=("discount_pct", "mean"),
        average_unit_price=("unit_selling_price", "mean")
    )
)

print("\nPromotion vs Non-Promotion Pricing")
print("=" * 70)

print(
    promotion_summary.to_string(index=False)
)

# ============================================================
# CATEGORY PRICING SUMMARY
# ============================================================

category_summary = (
    sales_pricing
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        sales_lines=("sales_line_id", "count"),
        average_list_price=("unit_price", "mean"),
        average_selling_price=("unit_selling_price", "mean"),
        average_discount=("discount_pct", "mean"),
        average_cost=("cost_price", "mean")
    )
    .sort_values(
        "average_selling_price",
        ascending=False
    )
)

print("\nCategory Pricing Summary")
print("=" * 70)

print(
    category_summary.to_string(index=False)
)

# ============================================================
# FINAL BASIC VALIDATION
# ============================================================

print("\nFinal Validation")
print("=" * 70)

print(
    "Sales rows after pricing merge:",
    len(sales_pricing)
)

print(
    "Missing product matches:",
    sales_pricing["category"].isna().sum()
)

if (
    len(sales_pricing) == 2_118_200
    and sales_pricing["category"].notna().all()
    and sales_pricing["unit_selling_price"].notna().all()
    and sales_pricing["cost_price"].notna().all()
    and sales_pricing["discount_pct"].notna().all()
    and (sales_pricing["unit_selling_price"] > 0).all()
    and (sales_pricing["cost_price"] > 0).all()
    and (
        (sales_pricing["discount_pct"] >= 0)
        &
        (sales_pricing["discount_pct"] <= 100)
    ).all()
):
    print("\n02.14 Step 1 validation: PASSED")
else:
    print("\n02.14 Step 1 validation: FAILED")

In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.14 STEP 2 — CREATE PRICING CONTROL PLAN
# ============================================================

BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data" / "synthetic"

OUTPUT_DIR = (
    DATA_DIR
    / "pricing_discount_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRODUCTS_FILE = DATA_DIR / "products_master.csv"

# ------------------------------------------------------------
# Load product master
# ------------------------------------------------------------

products = pd.read_csv(PRODUCTS_FILE)

print("02.14 STEP 2 — PRICING CONTROL PLAN")
print("=" * 70)

print("Products loaded:", len(products))

# ------------------------------------------------------------
# Select pricing fields
# ------------------------------------------------------------

pricing_plan = products[
    [
        "product_id",
        "category",
        "subcategory",
        "brand",
        "unit_price",
        "cost_price",
        "profit_margin_pct",
        "status"
    ]
].copy()

# ------------------------------------------------------------
# Convert numeric fields
# ------------------------------------------------------------

pricing_plan["unit_price"] = pd.to_numeric(
    pricing_plan["unit_price"],
    errors="coerce"
)

pricing_plan["cost_price"] = pd.to_numeric(
    pricing_plan["cost_price"],
    errors="coerce"
)

pricing_plan["profit_margin_pct"] = pd.to_numeric(
    pricing_plan["profit_margin_pct"],
    errors="coerce"
)

# ------------------------------------------------------------
# Create pricing rules
# ------------------------------------------------------------

pricing_plan["minimum_valid_selling_price"] = (
    pricing_plan["cost_price"]
)

pricing_plan["maximum_discount_pct"] = 60.0

pricing_plan["pricing_relationship_valid"] = (
    pricing_plan["unit_price"]
    >= pricing_plan["cost_price"]
)

# ------------------------------------------------------------
# Calculate available gross margin
# ------------------------------------------------------------

pricing_plan["gross_margin_value"] = (
    pricing_plan["unit_price"]
    - pricing_plan["cost_price"]
)

pricing_plan["gross_margin_pct_calculated"] = (
    pricing_plan["gross_margin_value"]
    / pricing_plan["unit_price"]
    * 100
)

# ============================================================
# VALIDATION
# ============================================================

print("\nPricing Control Validation")
print("=" * 70)

print(
    "Missing product IDs:",
    pricing_plan["product_id"].isna().sum()
)

print(
    "Duplicate product IDs:",
    pricing_plan["product_id"].duplicated().sum()
)

print(
    "Missing unit prices:",
    pricing_plan["unit_price"].isna().sum()
)

print(
    "Missing cost prices:",
    pricing_plan["cost_price"].isna().sum()
)

print(
    "Non-positive unit prices:",
    (pricing_plan["unit_price"] <= 0).sum()
)

print(
    "Non-positive cost prices:",
    (pricing_plan["cost_price"] <= 0).sum()
)

print(
    "Cost >= unit price:",
    (
        pricing_plan["cost_price"]
        >= pricing_plan["unit_price"]
    ).sum()
)

print(
    "Invalid calculated margins:",
    (
        pricing_plan["gross_margin_pct_calculated"] < 0
    ).sum()
)

# ============================================================
# CATEGORY PRICING SUMMARY
# ============================================================

category_summary = (
    pricing_plan
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        products=("product_id", "nunique"),
        average_unit_price=("unit_price", "mean"),
        average_cost_price=("cost_price", "mean"),
        average_margin_pct=(
            "gross_margin_pct_calculated",
            "mean"
        )
    )
)

print("\nCategory Pricing Control Summary")
print("=" * 70)

print(
    category_summary.to_string(index=False)
)

# ============================================================
# SAVE CONTROL PLAN
# ============================================================

CONTROL_PLAN_FILE = (
    OUTPUT_DIR
    / "pricing_control_plan.csv"
)

pricing_plan.to_csv(
    CONTROL_PLAN_FILE,
    index=False
)

print("\nPricing control plan saved to:")
print(CONTROL_PLAN_FILE)

# ============================================================
# FINAL VALIDATION
# ============================================================

if (
    len(pricing_plan) == len(products)
    and pricing_plan["product_id"].notna().all()
    and pricing_plan["product_id"].nunique() == len(pricing_plan)
    and pricing_plan["unit_price"].notna().all()
    and pricing_plan["cost_price"].notna().all()
    and (pricing_plan["unit_price"] > 0).all()
    and (pricing_plan["cost_price"] > 0).all()
    and (
        pricing_plan["cost_price"]
        < pricing_plan["unit_price"]
    ).all()
):
    print("\n02.14 Step 2 validation: PASSED")
else:
    print("\n02.14 Step 2 validation: FAILED")

02.14 STEP 2 — PRICING CONTROL PLAN
Products loaded: 2500

Pricing Control Validation
Missing product IDs: 0
Duplicate product IDs: 0
Missing unit prices: 0
Missing cost prices: 0
Non-positive unit prices: 0
Non-positive cost prices: 0
Cost >= unit price: 0
Invalid calculated margins: 0

Category Pricing Control Summary
              category  products  average_unit_price  average_cost_price  average_margin_pct
Beauty & Personal Care       505        31643.905010        25409.441545           19.493722
           Electronics       500        31446.949460        25298.801060           19.272204
               Fashion       509        30959.845560        24834.135442           19.522336
               Grocery       482        32275.378797        25725.828921           19.772933
        Home & Kitchen       504        28559.002560        22832.841071           19.688780

Pricing control plan saved to:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\pricing_discount_adju

In [5]:
import pandas as pd
from pathlib import Path
import numpy as np

# ============================================================
# 02.14 STEP 3 — SAFE PRICING & DISCOUNT LOGIC
# ============================================================

BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data" / "synthetic"

CONTROL_PLAN_FILE = (
    DATA_DIR
    / "pricing_discount_adjusted"
    / "pricing_control_plan.csv"
)

OUTPUT_DIR = (
    DATA_DIR
    / "pricing_discount_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load pricing control plan
# ------------------------------------------------------------

pricing_plan = pd.read_csv(CONTROL_PLAN_FILE)

print("02.14 STEP 3 — SAFE PRICING & DISCOUNT LOGIC")
print("=" * 70)

print("Pricing plan products:", len(pricing_plan))

# ------------------------------------------------------------
# Load sales files
# ------------------------------------------------------------

sales_files = sorted(DATA_DIR.glob("sales_*.csv"))

print("Sales files:", len(sales_files))

total_rows = 0
total_price_adjusted = 0
total_discount_adjusted = 0
total_cost_breaches = 0

monthly_results = []

# ------------------------------------------------------------
# Process each monthly file
# ------------------------------------------------------------

for file in sales_files:

    sales = pd.read_csv(file)

    original_rows = len(sales)
    total_rows += original_rows

    # --------------------------------------------------------
    # Numeric conversion
    # --------------------------------------------------------

    sales["unit_selling_price"] = pd.to_numeric(
        sales["unit_selling_price"],
        errors="coerce"
    )

    sales["discount_pct"] = pd.to_numeric(
        sales["discount_pct"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Attach product cost
    # --------------------------------------------------------

    sales = sales.merge(
        pricing_plan[
            [
                "product_id",
                "unit_price",
                "cost_price"
            ]
        ],
        on="product_id",
        how="left"
    )

    # --------------------------------------------------------
    # Ensure valid base selling price
    # --------------------------------------------------------

    original_price = sales["unit_selling_price"].copy()

    sales["unit_selling_price"] = (
        sales["unit_selling_price"]
        .fillna(sales["cost_price"])
    )

    # Selling price cannot be below cost
    sales["unit_selling_price"] = (
        sales[
            [
                "unit_selling_price",
                "cost_price"
            ]
        ]
        .max(axis=1)
    )

    price_adjusted_mask = (
        original_price
        < sales["unit_selling_price"]
    )

    price_adjusted_count = price_adjusted_mask.sum()

    total_price_adjusted += price_adjusted_count

    # --------------------------------------------------------
    # Calculate maximum safe discount
    # --------------------------------------------------------

    sales["maximum_safe_discount_pct"] = (
        (
            sales["unit_selling_price"]
            - sales["cost_price"]
        )
        / sales["unit_selling_price"]
        * 100
    )

    sales["maximum_safe_discount_pct"] = (
        sales["maximum_safe_discount_pct"]
        .clip(lower=0, upper=60)
    )

    # --------------------------------------------------------
    # Clean original discount
    # --------------------------------------------------------

    original_discount = sales["discount_pct"].copy()

    sales["discount_pct"] = (
        sales["discount_pct"]
        .fillna(0)
        .clip(lower=0, upper=60)
    )

    # --------------------------------------------------------
    # Apply safe discount
    # --------------------------------------------------------

    sales["discount_pct"] = (
        sales[
            [
                "discount_pct",
                "maximum_safe_discount_pct"
            ]
        ]
        .min(axis=1)
    )

    # Round DOWN to two decimal places.
    # This prevents rounding upward beyond the safe discount.
    sales["discount_pct"] = (
        np.floor(
            sales["discount_pct"] * 100
        ) / 100
    )

    discount_adjusted_mask = (
        original_discount
        .fillna(0)
        > sales["discount_pct"]
    )

    discount_adjusted_count = (
        discount_adjusted_mask.sum()
    )

    total_discount_adjusted += (
        discount_adjusted_count
    )

    # --------------------------------------------------------
    # Calculate effective selling price
    # --------------------------------------------------------

    sales["effective_unit_price"] = (
        sales["unit_selling_price"]
        * (
            1
            - sales["discount_pct"] / 100
        )
    )

    # --------------------------------------------------------
    # Final cost validation
    # --------------------------------------------------------

    cost_breach = (
        sales["effective_unit_price"]
        < sales["cost_price"]
    )

    cost_breach_count = cost_breach.sum()

    total_cost_breaches += cost_breach_count

    # --------------------------------------------------------
    # Remove temporary columns
    # --------------------------------------------------------

    sales = sales.drop(
        columns=[
            "unit_price",
            "cost_price",
            "maximum_safe_discount_pct",
            "effective_unit_price"
        ]
    )

    # --------------------------------------------------------
    # Save adjusted file
    # --------------------------------------------------------

    output_file = OUTPUT_DIR / file.name

    sales.to_csv(
        output_file,
        index=False
    )

    monthly_results.append({
        "file": file.name,
        "rows": len(sales),
        "price_adjusted": price_adjusted_count,
        "discounts_adjusted": discount_adjusted_count,
        "cost_breaches": cost_breach_count
    })


# ============================================================
# FINAL SUMMARY
# ============================================================

results = pd.DataFrame(
    monthly_results
)

print("\nMonthly Adjustment Summary")
print("=" * 70)

print(
    results.to_string(index=False)
)

print("\nOverall Results")
print("=" * 70)

print(
    "Total sales rows:",
    total_rows
)

print(
    "Selling prices adjusted:",
    total_price_adjusted
)

print(
    "Discounts adjusted:",
    total_discount_adjusted
)

print(
    "Remaining cost breaches:",
    total_cost_breaches
)

print("\nAdjusted sales files saved to:")
print(OUTPUT_DIR)

# ============================================================
# FINAL VALIDATION
# ============================================================

actual_files = len(
    list(
        OUTPUT_DIR.glob("sales_*.csv")
    )
)

if (
    total_rows == 2_118_200
    and len(sales_files) == 32
    and actual_files == 32
    and total_cost_breaches == 0
):
    print(
        "\n02.14 Step 3 validation: PASSED"
    )
else:
    print(
        "\n02.14 Step 3 validation: FAILED"
    )

02.14 STEP 3 — SAFE PRICING & DISCOUNT LOGIC
Pricing plan products: 2500
Sales files: 32

Monthly Adjustment Summary
             file  rows  price_adjusted  discounts_adjusted  cost_breaches
sales_2024-01.csv 53200               0                   0              0
sales_2024-02.csv 50400               0                   0              0
sales_2024-03.csv 56000               0                   0              0
sales_2024-04.csv 57100               0                   0              0
sales_2024-05.csv 58800               0                   0              0
sales_2024-06.csv 60500               0                   0              0
sales_2024-07.csv 57700               0                   0              0
sales_2024-08.csv 60500               0                   0              0
sales_2024-09.csv 64400               0                   0              0
sales_2024-10.csv 75600               0                   0              0
sales_2024-11.csv 81200               0                   

In [7]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.14 STEP 4 — PROMOTION & DISCOUNT RELATIONSHIP VALIDATION
# ============================================================

BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data" / "synthetic"

ADJUSTED_SALES_DIR = (
    DATA_DIR
    / "pricing_discount_adjusted"
    / "sales_adjusted"
)

PROMOTIONS_FILE = (
    DATA_DIR
    / "promotions_master.csv"
)

print("02.14 STEP 4 — PROMOTION & DISCOUNT VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Load promotion master
# ------------------------------------------------------------

promotions = pd.read_csv(
    PROMOTIONS_FILE
)

promotion_master_ids = set(
    promotions["promotion_id"]
    .astype(str)
    .str.strip()
)

print(
    "Promotion master rows:",
    len(promotions)
)

print(
    "Promotion master IDs:",
    len(promotion_master_ids)
)

# ------------------------------------------------------------
# Load adjusted sales files
# ------------------------------------------------------------

sales_files = sorted(
    ADJUSTED_SALES_DIR.glob("sales_*.csv")
)

sales_list = []

for file in sales_files:

    df = pd.read_csv(file)

    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

print(
    "Adjusted sales files loaded:",
    len(sales_files)
)

print(
    "Adjusted sales rows:",
    len(sales)
)

# ------------------------------------------------------------
# Clean promotion ID
# ------------------------------------------------------------

sales["promotion_id_clean"] = (
    sales["promotion_id"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# Identify blank / non-promotion rows
# ------------------------------------------------------------

blank_promotion_mask = (
    sales["promotion_id"].isna()
    |
    sales["promotion_id_clean"].isin(
        ["", "nan", "none", "null"]
    )
)

# ------------------------------------------------------------
# Identify actual promotion rows
# ------------------------------------------------------------

promotion_mask = ~blank_promotion_mask

sales["promotion_flag"] = promotion_mask

# ------------------------------------------------------------
# Validate ONLY non-blank promotion IDs
# ------------------------------------------------------------

invalid_promotion_mask = (
    promotion_mask
    &
    ~sales["promotion_id_clean"].isin(
        promotion_master_ids
    )
)

invalid_promotion_count = (
    invalid_promotion_mask.sum()
)

valid_promotion_count = (
    promotion_mask.sum()
    - invalid_promotion_count
)

# ============================================================
# DISCOUNT PREPARATION
# ============================================================

sales["discount_pct"] = pd.to_numeric(
    sales["discount_pct"],
    errors="coerce"
)

sales["discount_applied"] = (
    sales["discount_pct"] > 0
)

# ============================================================
# PROMOTION ID VALIDATION
# ============================================================

print("\nPromotion ID Validation")
print("=" * 70)

print(
    "Promotion rows:",
    promotion_mask.sum()
)

print(
    "Non-promotion rows:",
    blank_promotion_mask.sum()
)

print(
    "Valid promotion IDs:",
    valid_promotion_count
)

print(
    "Invalid promotion IDs:",
    invalid_promotion_count
)

# ============================================================
# PROMOTION VS NON-PROMOTION SUMMARY
# ============================================================

promotion_summary = (
    sales
    .groupby(
        "promotion_flag",
        as_index=False
    )
    .agg(
        sales_lines=(
            "sales_line_id",
            "count"
        ),
        orders=(
            "order_id",
            "nunique"
        ),
        average_discount=(
            "discount_pct",
            "mean"
        ),
        median_discount=(
            "discount_pct",
            "median"
        ),
        discounted_lines=(
            "discount_applied",
            "sum"
        )
    )
)

promotion_summary["discounted_line_pct"] = (
    promotion_summary["discounted_lines"]
    / promotion_summary["sales_lines"]
    * 100
)

print("\nPromotion vs Non-Promotion Summary")
print("=" * 70)

print(
    promotion_summary.to_string(
        index=False
    )
)

# ============================================================
# DISCOUNT VALIDATION
# ============================================================

print("\nDiscount Validation")
print("=" * 70)

print(
    "Missing discounts:",
    sales["discount_pct"].isna().sum()
)

print(
    "Negative discounts:",
    (
        sales["discount_pct"] < 0
    ).sum()
)

print(
    "Discounts above 60%:",
    (
        sales["discount_pct"] > 60
    ).sum()
)

# ============================================================
# PROMOTION-LEVEL SUMMARY
# ============================================================

promotion_detail = (
    sales[
        promotion_mask
    ]
    .groupby(
        "promotion_id_clean",
        as_index=False
    )
    .agg(
        sales_lines=(
            "sales_line_id",
            "count"
        ),
        orders=(
            "order_id",
            "nunique"
        ),
        average_discount=(
            "discount_pct",
            "mean"
        ),
        minimum_discount=(
            "discount_pct",
            "min"
        ),
        maximum_discount=(
            "discount_pct",
            "max"
        )
    )
    .sort_values(
        "sales_lines",
        ascending=False
    )
)

print("\nPromotion-Level Discount Summary")
print("=" * 70)

print(
    promotion_detail.head(20).to_string(
        index=False
    )
)

# ============================================================
# FINAL VALIDATION
# ============================================================

print("\nFinal Validation")
print("=" * 70)

print(
    "Adjusted sales rows:",
    len(sales)
)

print(
    "Adjusted sales files:",
    len(sales_files)
)

print(
    "Invalid promotion IDs:",
    invalid_promotion_count
)

print(
    "Missing discounts:",
    sales["discount_pct"].isna().sum()
)

print(
    "Discounts outside 0-60%:",
    (
        (sales["discount_pct"] < 0)
        |
        (sales["discount_pct"] > 60)
    ).sum()
)

if (
    len(sales) == 2_118_200
    and len(sales_files) == 32
    and invalid_promotion_count == 0
    and sales["discount_pct"].notna().all()
    and (
        sales["discount_pct"] >= 0
    ).all()
    and (
        sales["discount_pct"] <= 60
    ).all()
):
    print(
        "\n02.14 Step 4 validation: PASSED"
    )
else:
    print(
        "\n02.14 Step 4 validation: FAILED"
    )

02.14 STEP 4 — PROMOTION & DISCOUNT VALIDATION
Promotion master rows: 72
Promotion master IDs: 72
Adjusted sales files loaded: 32
Adjusted sales rows: 2118200

Promotion ID Validation
Promotion rows: 146225
Non-promotion rows: 1971975
Valid promotion IDs: 146225
Invalid promotion IDs: 0

Promotion vs Non-Promotion Summary
 promotion_flag  sales_lines  orders  average_discount  median_discount  discounted_lines  discounted_line_pct
          False      1971975 1610872           0.00000              0.0                 0                  0.0
           True       146225  142201          13.38759             12.3            146225                100.0

Discount Validation
Missing discounts: 0
Negative discounts: 0
Discounts above 60%: 0

Promotion-Level Discount Summary
promotion_id_clean  sales_lines  orders  average_discount  minimum_discount  maximum_discount
        PROMO00023         7498    7428         12.055788              9.22             12.30
        PROMO00011         5947   

In [8]:
import pandas as pd
from pathlib import Path

# ============================================================
# 02.14 STEP 5 — FINAL PRICING & DISCOUNT VALIDATION
# ============================================================

BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data" / "synthetic"

ADJUSTED_SALES_DIR = (
    DATA_DIR
    / "pricing_discount_adjusted"
    / "sales_adjusted"
)

PRICING_PLAN_FILE = (
    DATA_DIR
    / "pricing_discount_adjusted"
    / "pricing_control_plan.csv"
)

# ------------------------------------------------------------
# Load pricing plan
# ------------------------------------------------------------

pricing_plan = pd.read_csv(
    PRICING_PLAN_FILE
)

# ------------------------------------------------------------
# Load adjusted sales
# ------------------------------------------------------------

sales_files = sorted(
    ADJUSTED_SALES_DIR.glob("sales_*.csv")
)

sales_list = []

for file in sales_files:
    sales_list.append(
        pd.read_csv(file)
    )

sales = pd.concat(
    sales_list,
    ignore_index=True
)

# ------------------------------------------------------------
# Attach product cost
# ------------------------------------------------------------

sales = sales.merge(
    pricing_plan[
        [
            "product_id",
            "category",
            "cost_price"
        ]
    ],
    on="product_id",
    how="left"
)

# ------------------------------------------------------------
# Numeric conversion
# ------------------------------------------------------------

sales["unit_selling_price"] = pd.to_numeric(
    sales["unit_selling_price"],
    errors="coerce"
)

sales["cost_price"] = pd.to_numeric(
    sales["cost_price"],
    errors="coerce"
)

sales["discount_pct"] = pd.to_numeric(
    sales["discount_pct"],
    errors="coerce"
)

sales["quantity"] = pd.to_numeric(
    sales["quantity"],
    errors="coerce"
)

# ============================================================
# CALCULATE FINAL PRICING METRICS
# ============================================================

sales["effective_unit_price"] = (
    sales["unit_selling_price"]
    * (
        1
        - sales["discount_pct"] / 100
    )
)

sales["gross_sales_value"] = (
    sales["quantity"]
    * sales["unit_selling_price"]
)

sales["discount_value"] = (
    sales["gross_sales_value"]
    * sales["discount_pct"]
    / 100
)

sales["net_sales_value"] = (
    sales["gross_sales_value"]
    - sales["discount_value"]
)

# ============================================================
# FINAL VALIDATION
# ============================================================

print("02.14 STEP 5 — FINAL PRICING & DISCOUNT VALIDATION")
print("=" * 70)

print("\nData Volume")
print("-" * 70)

print("Sales files:", len(sales_files))
print("Sales rows:", len(sales))

print("\nPricing Validation")
print("-" * 70)

print(
    "Missing product cost:",
    sales["cost_price"].isna().sum()
)

print(
    "Missing selling price:",
    sales["unit_selling_price"].isna().sum()
)

print(
    "Missing discount:",
    sales["discount_pct"].isna().sum()
)

print(
    "Non-positive selling price:",
    (
        sales["unit_selling_price"] <= 0
    ).sum()
)

print(
    "Non-positive cost:",
    (
        sales["cost_price"] <= 0
    ).sum()
)

print(
    "Discount below 0%:",
    (
        sales["discount_pct"] < 0
    ).sum()
)

print(
    "Discount above 60%:",
    (
        sales["discount_pct"] > 60
    ).sum()
)

print(
    "Effective price below cost:",
    (
        sales["effective_unit_price"]
        < sales["cost_price"]
    ).sum()
)

print(
    "Negative net sales:",
    (
        sales["net_sales_value"] < 0
    ).sum()
)

print(
    "Zero/negative quantity:",
    (
        sales["quantity"] <= 0
    ).sum()
)

# ============================================================
# CATEGORY SUMMARY
# ============================================================

category_summary = (
    sales
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        sales_lines=(
            "sales_line_id",
            "count"
        ),
        average_selling_price=(
            "unit_selling_price",
            "mean"
        ),
        average_cost=(
            "cost_price",
            "mean"
        ),
        average_discount=(
            "discount_pct",
            "mean"
        ),
        total_gross_sales=(
            "gross_sales_value",
            "sum"
        ),
        total_discount_value=(
            "discount_value",
            "sum"
        ),
        total_net_sales=(
            "net_sales_value",
            "sum"
        )
    )
)

print("\nCategory Pricing Summary")
print("=" * 70)

print(
    category_summary.to_string(
        index=False
    )
)

# ============================================================
# FINAL VALIDATION
# ============================================================

cost_breaches = (
    sales["effective_unit_price"]
    < sales["cost_price"]
).sum()

negative_net_sales = (
    sales["net_sales_value"] < 0
).sum()

if (
    len(sales_files) == 32
    and len(sales) == 2_118_200
    and sales["cost_price"].notna().all()
    and sales["unit_selling_price"].notna().all()
    and sales["discount_pct"].notna().all()
    and sales["quantity"].notna().all()
    and (
        sales["unit_selling_price"] > 0
    ).all()
    and (
        sales["cost_price"] > 0
    ).all()
    and (
        sales["discount_pct"] >= 0
    ).all()
    and (
        sales["discount_pct"] <= 60
    ).all()
    and cost_breaches == 0
    and negative_net_sales == 0
    and (
        sales["quantity"] > 0
    ).all()
):
    print(
        "\n02.14 Step 5 validation: PASSED"
    )
else:
    print(
        "\n02.14 Step 5 validation: FAILED"
    )

02.14 STEP 5 — FINAL PRICING & DISCOUNT VALIDATION

Data Volume
----------------------------------------------------------------------
Sales files: 32
Sales rows: 2118200

Pricing Validation
----------------------------------------------------------------------
Missing product cost: 0
Missing selling price: 0
Missing discount: 0
Non-positive selling price: 0
Non-positive cost: 0
Discount below 0%: 0
Discount above 60%: 0
Effective price below cost: 0
Negative net sales: 0
Zero/negative quantity: 0

Category Pricing Summary
              category  sales_lines  average_selling_price  average_cost  average_discount  total_gross_sales  total_discount_value  total_net_sales
Beauty & Personal Care       434738           31348.196996  25156.668059          0.919230       2.178613e+10          2.011694e+08     2.158496e+10
           Electronics       440090           31411.641168  25289.036171          0.963359       2.209668e+10          2.137777e+08     2.188290e+10
               Fashion  